# Model Performance and Assumption Checks

This notebook reviews saved model performance reports and runs sanity checks for the ICU LOS classification pipeline. It uses aggregate report artifacts plus the synthetic sample model, so it can run without restricted MIMIC-IV data.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.data.splitting import assert_patient_split_integrity
from src.data.target import TARGET_COLUMN, TARGET_LABELS
from src.data.validation import assert_matching_feature_columns, assert_no_leakage_columns

reports_dir = REPO_ROOT / 'reports' / 'classification'
sample_path = REPO_ROOT / 'data' / 'sample' / 'icu_los_classification_sample.csv'
sample_model_path = REPO_ROOT / 'models' / 'icu_los_classifier_sample.joblib'

assert reports_dir.exists(), f'Missing reports directory: {reports_dir}'
assert sample_path.exists(), f'Missing sample data: {sample_path}'
assert sample_model_path.exists(), f'Missing sample model: {sample_model_path}'

## Performance Summary

In [ ]:
comparison = pd.read_csv(reports_dir / 'model_comparison.csv')
test_comparison = (
    comparison[comparison['split'].eq('test')]
    .sort_values('macro_f1', ascending=False)
    .reset_index(drop=True)
)
display(test_comparison)

best_model = test_comparison.loc[0, 'model']
assert best_model == 'icu_los_classifier', f'Expected selected model to lead by macro F1, got {best_model}'

In [ ]:
per_class = pd.read_csv(reports_dir / 'icu_los_classifier' / 'test_per_class_metrics.csv')
confusion = pd.read_csv(reports_dir / 'icu_los_classifier' / 'test_confusion_matrix.csv', index_col=0)
display(per_class)
display(confusion)

assert set(per_class['class']) == {0, 1, 2}
assert confusion.shape == (3, 3)
assert int(per_class['support'].sum()) == int(confusion.to_numpy().sum())

## Split and Leakage Checks

In [ ]:
split_df = pd.read_csv(reports_dir / 'patient_level_split.csv')
assert_patient_split_integrity(split_df)
assert set(split_df['split']) == {'train', 'val', 'test'}
assert set(split_df[TARGET_COLUMN].dropna().unique()).issubset({0, 1, 2})

split_summary = (
    split_df.groupby('split')
    .agg(rows=('stay_id', 'size'), subjects=('subject_id', 'nunique'))
    .sort_index()
)
display(split_summary)

In [ ]:
artifact = joblib.load(sample_model_path)
model = artifact['model']
feature_columns = artifact['feature_columns']
numeric_columns = artifact['numeric_columns']
categorical_columns = artifact['categorical_columns']

assert artifact['target_column'] == TARGET_COLUMN
assert artifact['target_labels'] == TARGET_LABELS
assert_no_leakage_columns(feature_columns)
assert set(numeric_columns).isdisjoint(set(categorical_columns))
assert set(numeric_columns).union(categorical_columns) == set(feature_columns)
assert type(model.named_steps['model']).__name__ == 'HistGradientBoostingClassifier'

print(f'Checked {len(feature_columns)} feature columns for leakage-prone names.')
print(type(model.named_steps['model']).__name__)

## Saved Model Smoke Test

In [ ]:
sample_df = pd.read_csv(sample_path)
missing = sorted(set(feature_columns).difference(sample_df.columns))
extra = sorted(set(sample_df.columns).difference(feature_columns + ['subject_id', 'hadm_id', 'stay_id', 'intime', TARGET_COLUMN]))
assert not missing, f'Sample data is missing model features: {missing}'
assert_matching_feature_columns(feature_columns, list(sample_df[feature_columns].columns))

pred = model.predict(sample_df[feature_columns])
proba = model.predict_proba(sample_df[feature_columns])
assert len(pred) == len(sample_df)
assert proba.shape == (len(sample_df), 3)
assert set(pred).issubset({0, 1, 2})

predictions = sample_df[['subject_id', 'stay_id', TARGET_COLUMN]].copy()
predictions['predicted_los_category'] = pred
for idx, label in enumerate(model.classes_):
    predictions[f'prob_class_{label}'] = proba[:, idx]
display(predictions.head(10))

## Check Summary

In [ ]:
checks = pd.DataFrame(
    [
        {'check': 'patient split integrity', 'status': 'passed'},
        {'check': 'target labels limited to 0/1/2', 'status': 'passed'},
        {'check': 'feature leakage column scan', 'status': 'passed'},
        {'check': 'numeric/categorical feature partition', 'status': 'passed'},
        {'check': 'sample feature matrix matches saved model', 'status': 'passed'},
        {'check': 'saved model prediction/probability smoke test', 'status': 'passed'},
    ]
)
display(checks)